# Шаг 3. Data drift и деградация модели через EvidentlyAI

Цель — продемонстрировать, что при сдвиге распределений входных признаков (covariate shift) модель, обученная на исходных данных, теряет качество. Для этого:

1. Берём датасет Wine (sklearn, 13 числовых признаков, 3 класса).
2. Делим его 50/50 на **reference** и **current**.
3. В **current** искусственно сдвигаем несколько признаков (умножение на коэффициент + Гауссов шум).
4. Обучаем `LogisticRegression` на reference.
5. Сравниваем метрики на reference и на current → деградация.
6. Запускаем `Report(DataDriftPreset, DataQualityPreset)` и `Report(ClassificationPreset)`.
7. Сохраняем HTML-отчёты и `metrics_summary.json`.

> Установка зависимостей (один раз):
> ```bash
> pip install -r requirements.txt
> ```

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

from evidently import ColumnMapping
from evidently.report import Report
from evidently.metric_preset import (
    DataDriftPreset,
    DataQualityPreset,
    ClassificationPreset,
)

RANDOM_STATE = 42
HERE = Path.cwd()

## 1. Готовим reference и current с искусственным drift

Сценарий — covariate shift: производственное оборудование начало мерить алкоголь, magnesium и color_intensity по-другому (откалибровали датчики, поменяли поставщика). Распределение целевых меток мы не трогаем — это именно **data drift**, а не concept drift.

In [ ]:
raw = load_wine(as_frame=True)
df = raw.frame.copy()
features = list(raw.feature_names)
target = 'target'

reference, current = train_test_split(
    df, test_size=0.5, random_state=RANDOM_STATE, stratify=df[target]
)

rng = np.random.default_rng(RANDOM_STATE)
current_drift = current.copy()
current_drift['alcohol'] *= 1.5
current_drift['color_intensity'] *= 2.0
current_drift['proline'] *= 0.4
current_drift['magnesium'] = current_drift['magnesium'] + rng.normal(20, 15, len(current_drift))
current_drift['malic_acid'] = current_drift['malic_acid'] + rng.normal(1.0, 0.6, len(current_drift))

print('reference:', reference.shape, 'current_drift:', current_drift.shape)
reference[features].describe().T[['mean', 'std']].join(
    current_drift[features].describe().T[['mean', 'std']], lsuffix='_ref', rsuffix='_cur'
)

## 2. Обучаем модель на reference и считаем метрики

In [ ]:
model = LogisticRegression(max_iter=5000)
model.fit(reference[features], reference[target])

pred_ref = model.predict(reference[features])
pred_cur = model.predict(current_drift[features])

perf_metrics = {
    'accuracy_reference': float(accuracy_score(reference[target], pred_ref)),
    'accuracy_current_drifted': float(accuracy_score(current_drift[target], pred_cur)),
    'f1_macro_reference': float(f1_score(reference[target], pred_ref, average='macro')),
    'f1_macro_current_drifted': float(f1_score(current_drift[target], pred_cur, average='macro')),
}
pd.Series(perf_metrics).to_frame('value')

Ожидаемо: на reference accuracy ≈ 1.0, а на смещённом current — заметное падение (~0.4). Это и есть **деградация модели**, возникшая из-за data drift.

In [ ]:
print('--- reference ---')
print(classification_report(reference[target], pred_ref, digits=3))
print('--- current (drifted) ---')
print(classification_report(current_drift[target], pred_cur, digits=3))

## 3. Отчёт о data drift

Evidently сравнит распределения по каждой колонке (KS-test для числовых, по умолчанию). Колонка считается «дрифтнувшей», если p-value ниже порога. На выходе — HTML, удобный для шаринга.

Параметр `drift_share=0.3` понижает порог: считаем dataset «дрифтнувшим» уже при 30 % смещённых фич (по умолчанию 0.5). Для нашего случая (6 из 13 фич) это критически важно, иначе агрегат `dataset_drift` останется `False`.

In [ ]:
ref_full = reference.copy(); ref_full['prediction'] = pred_ref
cur_full = current_drift.copy(); cur_full['prediction'] = pred_cur

column_mapping = ColumnMapping(
    target=target,
    prediction='prediction',
    numerical_features=features,
)

data_report = Report(metrics=[DataDriftPreset(drift_share=0.3), DataQualityPreset()])
data_report.run(reference_data=ref_full, current_data=cur_full, column_mapping=column_mapping)
data_report.save_html('data_drift_report.html')
data_report  # отображается inline в Jupyter

## 4. Отчёт о деградации модели (Classification)

Здесь Evidently сравнивает качество классификатора на reference и на current и подсвечивает, какие классы «просели».

In [ ]:
perf_report = Report(metrics=[ClassificationPreset()])
perf_report.run(reference_data=ref_full, current_data=cur_full, column_mapping=column_mapping)
perf_report.save_html('model_performance_report.html')
perf_report

## 5. Сводка и сохранение `metrics_summary.json`

Достаём из `data_report.as_dict()` агрегаты по data drift, складываем с метриками классификации и сохраняем в JSON — это удобный артефакт для отчёта и для дальнейшего пуша в Prometheus Pushgateway.

In [ ]:
drift_metrics = {}
for metric in data_report.as_dict()['metrics']:
    if metric.get('metric') == 'DatasetDriftMetric':
        r = metric['result']
        drift_metrics['drifted_columns'] = int(r.get('number_of_drifted_columns', 0))
        drift_metrics['share_of_drifted_columns'] = float(r.get('share_of_drifted_columns', 0.0))
        drift_metrics['dataset_drift'] = bool(r.get('dataset_drift', False))
        break

summary = {
    **perf_metrics,
    **drift_metrics,
    'data_drift_report': 'data_drift_report.html',
    'model_performance_report': 'model_performance_report.html',
}
(HERE / 'metrics_summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False))
summary

## 6. Что получили

Артефакты в текущей папке:
* `data_drift_report.html` — drift по 13 фичам.
* `model_performance_report.html` — падение accuracy/F1 по классам.
* `metrics_summary.json` — agg-метрики (accuracy, F1, drift share).

Что показывает эксперимент:
1. Сильный сдвиг распределений нескольких признаков **обнаруживается Evidently'ем** (KS-test, p-value < 0.05) → 6 колонок дрифтнувшие, share = 0.46.
2. Та же модель на тех же *метках*, но на сдвинутых *признаках* теряет accuracy с **1.0 до ~0.40** — это **деградация модели**, вызванная data drift, а не concept drift.
3. В проде такой отчёт надо запускать по расписанию (Airflow / cron / Argo) и оповещать команду, если `share_of_drifted_columns > 0.3` (например). Эту метрику также можно push'ить в Prometheus и подключить к тому же Alertmanager + Telegram, как мы делали для latency в Шаге 2.